# Train RF-DETR Nano on COCO2017 with a Cloud TPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/train-coco2017-tpu.ipynb)

Trains **RF-DETR Nano** on the full public **COCO2017** dataset on a single TPU chip, the same recipe as the
[CUDA COCO2017 cookbook](train-coco2017.ipynb) moved to Google's TPU hardware. Most PyTorch users have never
trained on a TPU, so this notebook explains the few ideas you need along the way, and keeps everything else
out.

Before you run anything, switch the Colab runtime to a TPU: **Runtime → Change runtime type → TPU**. This
notebook targets one TPU chip, the size Colab's TPU runtimes provide.

> **Note:** No epoch time, memory figure or mAP is quoted here, because none has been measured for this recipe
> on TPU yet. Section 5 shows how to read throughput off the first epochs yourself.

## How a TPU differs from a GPU

PyTorch reaches a TPU through **PyTorch/XLA** (`torch_xla`). Three ideas explain every setting below:

1. **Lazy execution.** A GPU runs each operation as soon as Python calls it. On XLA, operations only record
   a graph. At the end of each training step the graph is compiled into one TPU program and run.
2. **Compile once per shape.** XLA compiles a separate program for each set of tensor shapes it sees. The
   first step is slow because it compiles. Later steps with the **same shapes** reuse that program and are
   fast. A batch with **new shapes** pays another compilation. Most of the TPU work is keeping shapes fixed.
3. **bfloat16 is native.** TPU matrix units compute in bf16. No gradient scaler is needed, unlike fp16 on a
   GPU.

In RF-DETR the shapes that change are the image size and the number of ground-truth boxes per image. Section
4 pins both.

## 1 - Install PyTorch/XLA and RF-DETR

`torch_xla` is built against **one** `torch` minor version. A mismatched pair fails at `import torch_xla` with
an `undefined symbol` error. RF-DETR's `xla` extra pins the pair it supports (`torch` 2.9 with `torch_xla`
2.9). The `[tpu]` extra of `torch_xla` adds `libtpu`, the runtime that talks to the TPU chip. `torchvision`
must match `torch` too, so it is pinned alongside.

The throughput fixes this notebook relies on are on the `develop` branch. Replace the git URL with a version
pin once a release containing them is out.

If pip replaces the preinstalled `torch`, restart the session (**Runtime → Restart session**) once the cell
finishes, then continue with section 2.

In [ ]:
%pip install --progress-bar on "torch==2.9.*" "torchvision==0.24.*" "torch_xla[tpu]==2.9.*"
%pip install --progress-bar on "rfdetr[train,visual,xla] @ git+https://github.com/roboflow/rf-detr.git@develop"

## 2 - Meet the TPU

`PJRT_DEVICE=TPU` tells PyTorch/XLA which backend to open. Set it before `torch_xla` is imported.
`torch_xla.device()` returns the TPU as an ordinary `torch.device`, of type `xla`.

In [ ]:
import os

os.environ.setdefault("PJRT_DEVICE", "TPU")

import torch
import torch_xla
import torch_xla.debug.metrics as met
import torch_xla.runtime as xr

print("torch:", torch.__version__, "| torch_xla:", torch_xla.__version__)
print("XLA device type:", xr.device_type(), "| devices:", xr.global_runtime_device_count())

DEVICE = torch_xla.device()
print("Device:", DEVICE)

A short experiment shows lazy execution and per-shape compilation. `torch_xla.sync()` marks the end of a
step: it compiles the recorded graph (if this shape is new) and runs it. The `CompileTime` metric counts
compilations.

The same shape twice compiles once. A new shape compiles again.

In [ ]:
def compile_count() -> int:
    """Return how many XLA graphs this process has compiled so far."""
    data = met.metric_data("CompileTime")
    return 0 if data is None else data[0]


def step(size: int) -> None:
    """Record one small matrix multiply on the TPU and run it."""
    x = torch.randn(size, size, device=DEVICE)
    y = (x @ x).sum()
    torch_xla.sync()
    print(f"{size}x{size}: result {y.item():.1f}, compilations so far: {compile_count()}")


step(256)  # new shape: compiles
step(256)  # same shape: reuses the compiled program
step(512)  # new shape: compiles again

Training follows the same rule. Each distinct batch shape costs one compilation, which can take minutes for
a detection model. A fixed shape compiles in the first steps and then runs at full speed.

## 3 - Download COCO2017

About 19 GB of images plus annotations. Downloading a zip and then running `unzip` writes the 18 GB training
archive to disk, reads it back, and writes its 118,287 images out again, which is slow on a Colab disk.
`bsdtar` (from `libarchive-tools`) instead reads the zip as a stream, so each archive is extracted while it
downloads and never lands on disk. The cost: an interrupted stream cannot resume and starts over. The check
below catches a broken run.

In [ ]:
!apt-get -qq install -y libarchive-tools
!mkdir -p datasets/coco2017
!cd datasets/coco2017 && wget -q --show-progress -O- http://images.cocodataset.org/zips/train2017.zip | bsdtar -xf -
!cd datasets/coco2017 && wget -q --show-progress -O- http://images.cocodataset.org/zips/val2017.zip | bsdtar -xf -
!cd datasets/coco2017 && wget -q --show-progress -O- http://images.cocodataset.org/annotations/annotations_trainval2017.zip | bsdtar -xf -
!df -h datasets/coco2017

Check the extraction now, so that a truncated download fails here and not hours into training.

In [ ]:
from pathlib import Path

COCO_ROOT = Path("datasets/coco2017")
EXPECTED_IMAGES = {"train2017": 118_287, "val2017": 5_000}

for split, expected in EXPECTED_IMAGES.items():
    found = sum(1 for _ in (COCO_ROOT / split).glob("*.jpg"))
    assert found == expected, f"{COCO_ROOT / split} holds {found} images, expected {expected}. Re-run the download."
    annotations = COCO_ROOT / "annotations" / f"instances_{split}.json"
    assert annotations.is_file(), f"Missing {annotations}. Re-run the download."

print("COCO2017 is complete.")

## 4 - Train

`pretrain_weights=None` starts the detector from random initialization, so this trains COCO2017 from scratch.
The DINOv2 backbone still loads its self-supervised weights. The `PretrainWeightsCompatibilityWarning` that
follows is expected. For the best accuracy on your own data, fine-tune from the default checkpoint instead.

Every TPU-specific argument of `model.train()` is listed here:

- **`device="xla"`** — required. RF-DETR picks its default device from what PyTorch reports as an
  accelerator, and on a TPU host that is `"cpu"`. Without this argument, training runs on the host CPU.
  `"xla"` selects PyTorch Lightning's TPU accelerator.
- **`amp_dtype="auto"`** — on the TPU accelerator this selects Lightning's `bf16-true` precision: weights and
  activations in bf16, with no gradient scaler.
- **`multi_scale=False`** and **`square_resize_div_64=True`** — every training image is resized to one
  fixed square, Nano's 384 px. The default multi-scale schedule would give XLA a new image size to compile
  for every few batches.
- **`pad_targets_to=100`** — the loss is shaped by the number of ground-truth boxes in each image, and COCO
  images hold anything from 0 to dozens. Without padding, nearly every batch has a new shape and recompiles.
  Padding each image's targets to 100 rows gives the loss one shape. Padded rows are masked out of the
  matching and the loss, so training is unchanged. An image with more than 100 real boxes loses its extra
  boxes, with a warning. Padding applies to the training loader only, so validation still scores real boxes.
- **`batch_size=32`** — an integer, never `"auto"`: the automatic batch probe measures CUDA memory. 32 is a
  starting point, not a measured optimum. If the TPU runs out of memory, halve it.

The training loader drops its last incomplete batch, so every training step has the same shape. Validation
keeps every image, so its final, smaller batch compiles once more.

Leave `compile` and `cuda_graphs` at their defaults (`False`). Both are CUDA features, and on a TPU the XLA
compiler already does that work.

The other settings match the CUDA cookbook: a cosine schedule with one warm-up epoch, since the default step
schedule never decays within 40 epochs, and reproducible seeding.

In [ ]:
from rfdetr import RFDETRNano

OUTPUT_DIR = "output/det_coco2017_nano_tpu"
EPOCHS = 40
BATCH_SIZE = 32  # Starting point; not measured on TPU. Halve on an out-of-memory error.
NUM_WORKERS = min(os.cpu_count() or 2, 16)  # COCO needs more JPEG decoders than the default 2.

model = RFDETRNano(pretrain_weights=None)  # type: ignore[no-untyped-call]

Colab ends idle or long sessions, so expect to resume. `last.ckpt` is written every epoch. To continue an
interrupted run, add `resume=f"{OUTPUT_DIR}/last.ckpt"` to the call below and run it again. A new session
starts with an empty disk: to keep checkpoints across sessions, mount Google Drive and point `OUTPUT_DIR` at it.

Expect the first steps to be slow while XLA compiles the training step, then a jump in speed. The first
validation pass compiles too. Judge throughput from the second epoch on.

In [ ]:
model.train(
    dataset_file="coco",
    dataset_dir=COCO_ROOT,
    output_dir=OUTPUT_DIR,
    device="xla",
    amp_dtype="auto",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=1,
    num_workers=NUM_WORKERS,
    multi_scale=False,
    square_resize_div_64=True,
    pad_targets_to=100,
    lr_scheduler="cosine",
    warmup_epochs=1,
    seed=0,
    tensorboard=False,
    progress_bar="tqdm",
)

## 5 - Check for recompilation

If training stays slow after the first epoch, the likely cause is recompilation: some shape still changes.
The `CompileTime` count from section 2 answers that. A healthy run compiles a small, fixed number of
programs: the training step, validation, and validation's smaller last batch. A count that grows with the
number of batches means a shape changes from batch to batch. Check that the arguments in section 4 are all
set. To check early, interrupt training after a few hundred steps and run this cell.

In [ ]:
print("Compilations:", compile_count())
print(met.short_metrics_report())

## 6 - Plot the metrics

`CSVLogger` writes losses and COCO mAP to `metrics.csv` in `OUTPUT_DIR` once per epoch.
`val/mAP_50_95` is the primary COCO metric.

In [ ]:
from IPython.display import display
from matplotlib import pyplot as plt

from rfdetr.visualize.training import plot_loss_metrics, plot_map_metrics

METRICS_CSV = f"{OUTPUT_DIR}/metrics.csv"

for figure in (plot_loss_metrics(METRICS_CSV), plot_map_metrics(METRICS_CSV)):
    display(figure)
    plt.close(figure)